# ASM2D-TSN Simulation

This notebook runs the ASM2D-TSN simulation model from the source package and persists the generated dataset and metadata under `data/asm2d-tsn/simulation`.

The reported output variables here are exclusively the configured composites: COD, TN, TKN, TP, and TSS. The dissolved-oxygen state follows the `S_O` notation used by the mechanistic model.

In [ ]:
import pandas as pd
from src.models.simulation.asm2d_tsn_simulation import (
    run_asm2d_tsn_simulation,
    sweep_asm2d_tsn_operating_space,
 )

simulation_result = run_asm2d_tsn_simulation(
    save_artifacts=True,
    include_debug_data=True,
    show_progress=True,
    progress_description="ASM2D-TSN dataset",
)

dataset = simulation_result["dataset"]
metadata = simulation_result["metadata"]
artifact_paths = simulation_result["artifact_paths"]
petersen_matrix = simulation_result["petersen_matrix"]
composition_matrix = simulation_result["composition_matrix"]
effluent_states = simulation_result["effluent_states"]
solver_diagnostics = simulation_result["solver_diagnostics"]
solver_summary = simulation_result["solver_summary"]

print(f"Generated {len(dataset)} rows for {metadata['simulation_name']}.")
print(f"Dataset saved to: {artifact_paths['dataset_csv']}")
print(f"Metadata saved to: {artifact_paths['metadata_json']}")
print(f"Petersen matrix shape: {petersen_matrix.shape}")
print(f"Composition matrix shape: {composition_matrix.shape}")
print("Saved dataset remains composite-only; full state diagnostics stay in memory.")

output_columns = [column_name for column_name in metadata["dependent_columns"] if column_name.startswith("Out_")]
print(f"Reported output variables: {', '.join(output_columns)}")

display(dataset.head())
display(pd.Series(metadata, name="value").to_frame())
display(pd.json_normalize(solver_summary, sep="."))
display(solver_diagnostics.head())
display(effluent_states.head())
display(pd.DataFrame(petersen_matrix, index=metadata["processes"], columns=metadata["state_columns"]))
display(pd.DataFrame(composition_matrix, index=metadata["measured_output_columns"], columns=metadata["state_columns"]))

In [ ]:
import numpy as np
import pandas as pd
from scipy.linalg import null_space

from src.utils.process import has_active_projection

icsor_constraint_basis = null_space(petersen_matrix)
icsor_A_matrix = icsor_constraint_basis.T

icsor_A_matrix = np.round(icsor_A_matrix, 5)
icsor_A_matrix[np.abs(icsor_A_matrix) < 1e-10] = 0.0

for row_index in range(icsor_A_matrix.shape[0]):
    non_zero_entries = icsor_A_matrix[row_index, icsor_A_matrix[row_index, :] != 0]
    if len(non_zero_entries) > 0:
        icsor_A_matrix[row_index, :] = icsor_A_matrix[row_index, :] / non_zero_entries[0]

macroscopic_stoichiometric_matrix = petersen_matrix @ composition_matrix.T
measured_constraint_basis = null_space(macroscopic_stoichiometric_matrix)
A_matrix = measured_constraint_basis.T

A_matrix = np.round(A_matrix, 5)
A_matrix[np.abs(A_matrix) < 1e-10] = 0.0

for row_index in range(A_matrix.shape[0]):
    non_zero_entries = A_matrix[row_index, A_matrix[row_index, :] != 0]
    if len(non_zero_entries) > 0:
        A_matrix[row_index, :] = A_matrix[row_index, :] / non_zero_entries[0]

classical_projection_active = has_active_projection(A_matrix)
measured_output_dimension = int(A_matrix.shape[1])
measured_nullity = int(A_matrix.shape[0])
macroscopic_rank = int(np.linalg.matrix_rank(macroscopic_stoichiometric_matrix))
classical_projection_status = pd.DataFrame(
    [
        {
            "projection_active": classical_projection_active,
            "constraint_status": "active" if classical_projection_active else "inactive_trivial_null_space",
            "measured_output_dimension": measured_output_dimension,
            "rank_S_macro": macroscopic_rank,
            "nullity_S_macro": measured_nullity,
        }
    ]
)

print(f"Fractional Petersen matrix shape: {petersen_matrix.shape}")
print(f"icsor invariant matrix shape: {icsor_A_matrix.shape}")
print(f"Measured-space invariant matrix shape kept for downstream regressors: {A_matrix.shape}")
if classical_projection_active:
    print("Classical measured-space projection remains active because the measured-output null space is non-trivial.")
else:
    print("Classical measured-space projection is inactive because null_space(nu I_comp^T) is trivial in the measured-output basis.")
    print("Projected classical results and measured-space mass-balance discrepancy tables are suppressed downstream.")

display(
    pd.DataFrame(
        icsor_A_matrix,
        index=[f"constraint_{index + 1}" for index in range(icsor_A_matrix.shape[0])],
        columns=metadata["state_columns"],
    )
)
display(
    pd.DataFrame(
        A_matrix,
        index=[f"constraint_{index + 1}" for index in range(A_matrix.shape[0])],
        columns=metadata["measured_output_columns"],
    )
)
display(classical_projection_status)

## Solver Sweep

This section samples the configured operating space using Latin Hypercube Sampling (LHS) without changing the saved dataset contract. It summarizes how often the current acceptance threshold is met, how often the multistart guess wins, and how often dynamic relaxation is needed.

In [ ]:
SWEEP_SAMPLE_COUNT = 1024

sweep_result = sweep_asm2d_tsn_operating_space(
    n_samples=SWEEP_SAMPLE_COUNT,
    random_seed=int(metadata["random_seed"]),
    show_progress=True,
    progress_description="ASM2D-TSN operating-space sweep",
)

sweep_summary = sweep_result["summary"]
sweep_diagnostics = sweep_result["solver_diagnostics"].copy()
sweep_diagnostics["threshold_margin"] = (
    sweep_diagnostics["acceptance_threshold"] - sweep_diagnostics["residual_max"]
)

print(f"Operating-space sweep completed for {sweep_summary['sample_count']} sampled points.")
print(
    f"Accepted points at threshold {sweep_summary['acceptance_threshold']}: "
    f"{sweep_summary['accepted_count']} ({sweep_summary['accepted_rate']:.1%})"
 )
print(f"Multistart selected rate: {sweep_summary['multistart_selected_rate']:.1%}")
print(f"Dynamic relaxation used rate: {sweep_summary['dynamic_relaxation_used_rate']:.1%}")
print(f"Dynamic relaxation improved rate: {sweep_summary['dynamic_relaxation_improved_rate']:.1%}")

display(pd.json_normalize(sweep_summary, sep="."))
display(sweep_diagnostics.sort_values("residual_max", ascending=False).head(20))